# DINO: Self-Distillation with No Labels

**Paper**: Caron et al., 2021 — *Emerging Properties in Self-Supervised Vision Transformers*

DINO (Self-**DI**stillation with **NO** labels) trains a **student** network to match the output distribution of a **teacher** network that is an exponential moving average (EMA) of the student. Unlike BYOL, DINO uses a **centering** trick to prevent collapse without negative pairs or batch normalization.

Key ideas:
- **Knowledge distillation without labels**: student predicts teacher's soft assignments
- **Multi-crop strategy**: 2 global crops (224×224) + N local crops (96×96); teacher sees only global
- **Centering**: subtract running mean from teacher logits — prevents one dimension dominating
- **Sharpening**: apply low temperature to teacher softmax — sharpen the target distribution
- **EMA teacher**: teacher weights = exponential moving average of student (τ ≈ 0.996)
- **No negative pairs, no predictor MLP asymmetry** (unlike BYOL)

```
x_global1, x_global2 → teacher → softmax(T_t) → centering → target
x_all_views          → student → softmax(T_s) → cross-entropy with target
```

<img src="../figures/dino_arch.png" width="800"/>

*DINO: student sees all crops, teacher (EMA) sees only global crops. Centering prevents collapse.*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Dataset
import numpy as np
import copy
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Multi-Crop Augmentation

DINO uses a **multi-crop** strategy:
- **2 global crops** (scale 0.4–1.0, size 224→32 for CIFAR) — used by both student and teacher
- **N local crops** (scale 0.05–0.4, size 96→16 for CIFAR) — used by student only

The teacher only processes global views, giving it a "broader" context. The student must predict the teacher's output even from small local patches — forcing it to learn globally consistent features.

In [ ]:
class DINOAugmentation:
    """Multi-crop augmentation for DINO. Returns (global_crops, local_crops)."""

    def __init__(self, n_local=4, global_size=32, local_size=16):
        color_jitter = T.ColorJitter(0.4, 0.4, 0.2, 0.1)

        self.global_transform = T.Compose([
            T.RandomResizedCrop(global_size, scale=(0.4, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomApply([color_jitter], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize([0.4914, 0.4822, 0.4465], [0.247, 0.243, 0.261]),
        ])

        self.local_transform = T.Compose([
            T.RandomResizedCrop(local_size, scale=(0.05, 0.4)),
            T.RandomHorizontalFlip(),
            T.RandomApply([color_jitter], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize([0.4914, 0.4822, 0.4465], [0.247, 0.243, 0.261]),
        ])

        self.n_local = n_local

    def __call__(self, x):
        global_crops = [self.global_transform(x), self.global_transform(x)]
        local_crops = [self.local_transform(x) for _ in range(self.n_local)]
        return global_crops, local_crops


class MultiCropDataset(Dataset):
    def __init__(self, base_dataset, transform):
        self.dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        global_crops, local_crops = self.transform(img)
        return global_crops, local_crops, label

## DINO Head

The **projection head** maps encoder features to a K-dimensional space (K = number of "prototype" dimensions, e.g. 2048 or 4096). It consists of:
- 3-layer MLP with GELU and L2-normalized bottleneck
- **Weight-normalized** final linear layer (no bias) — this helps training stability

The output is a vector of raw logits (not softmax). Softmax with different temperatures is applied later for student (high T) and teacher (low T).

In [ ]:
class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim=2048, hidden_dim=512, bottleneck_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, bottleneck_dim),
        )
        self.last_layer = nn.utils.weight_norm(nn.Linear(bottleneck_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1)
        self.last_layer.weight_g.requires_grad = False

    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=-1, p=2)
        x = self.last_layer(x)
        return x


class DINONet(nn.Module):
    """Encoder + DINO head wrapper."""
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head = head

    def forward(self, x):
        return self.head(self.encoder(x))

## DINO Loss

The loss is the cross-entropy between student and teacher soft probability distributions:

$$\mathcal{L} = -\sum_{x_t \in \{x^{g_1}, x^{g_2}\}} \sum_{\substack{x_s \in V \\ x_s \neq x_t}} P_t(x_t) \log P_s(x_s)$$

where:
- $P_s(x) = \text{softmax}(z_s(x) / \tau_s)$ — student with high temperature $\tau_s$ (e.g. 0.1)
- $P_t(x) = \text{softmax}((z_t(x) - c) / \tau_t)$ — teacher with low temperature $\tau_t$ (e.g. 0.04) and center $c$
- $c$ is a running mean of teacher outputs (EMA) — prevents collapse to one dimension

In [ ]:
class DINOLoss(nn.Module):
    def __init__(self, out_dim, student_temp=0.1, teacher_temp=0.04, center_momentum=0.9):
        super().__init__()
        self.student_temp = student_temp
        self.teacher_temp = teacher_temp
        self.center_momentum = center_momentum
        self.register_buffer('center', torch.zeros(1, out_dim))

    @torch.no_grad()
    def update_center(self, teacher_output):
        # teacher_output: (N, out_dim) — average over batch
        batch_center = teacher_output.mean(dim=0, keepdim=True)
        self.center = self.center_momentum * self.center + (1 - self.center_momentum) * batch_center

    def forward(self, student_out, teacher_out, n_global=2):
        """
        student_out: list of (N, out_dim) tensors for all views (global + local)
        teacher_out: list of (N, out_dim) tensors for global views only
        """
        # Sharpen teacher distribution with low temperature + centering
        teacher_probs = [
            F.softmax((t - self.center) / self.teacher_temp, dim=-1).detach()
            for t in teacher_out
        ]

        total_loss = 0.0
        n_pairs = 0

        for t_idx, t_prob in enumerate(teacher_probs):
            for s_idx, s_logit in enumerate(student_out):
                # Skip same view (teacher global i vs student global i)
                if s_idx == t_idx:
                    continue
                s_prob = F.log_softmax(s_logit / self.student_temp, dim=-1)
                loss = -(t_prob * s_prob).sum(dim=-1).mean()
                total_loss += loss
                n_pairs += 1

        total_loss /= n_pairs

        # Update center with teacher global outputs
        all_teacher = torch.cat(teacher_out, dim=0)
        self.update_center(all_teacher)

        return total_loss

## Build Student and Teacher

We use a lightweight ResNet-18 as the backbone for CIFAR-10. The teacher starts as an exact copy of the student — no gradients flow through it. Its weights are updated only via EMA:

$$\theta_t \leftarrow \tau \cdot \theta_t + (1 - \tau) \cdot \theta_s$$

The EMA momentum $\tau$ is typically annealed from 0.996 → 1.0 over training (cosine schedule) to stabilize later-stage training.

In [ ]:
def build_encoder():
    """ResNet-18 with identity fc, adapted for small images."""
    enc = torchvision.models.resnet18(weights=None)
    # Adapt for CIFAR (32x32): replace 7x7 conv with 3x3, remove maxpool
    enc.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    enc.maxpool = nn.Identity()
    feat_dim = enc.fc.in_features
    enc.fc = nn.Identity()
    return enc, feat_dim


OUT_DIM = 2048

student_enc, feat_dim = build_encoder()
student_head = DINOHead(feat_dim, out_dim=OUT_DIM)
student = DINONet(student_enc, student_head).to(device)

teacher_enc, _ = build_encoder()
teacher_head = DINOHead(feat_dim, out_dim=OUT_DIM)
teacher = DINONet(teacher_enc, teacher_head).to(device)

# Initialize teacher = student, freeze teacher
teacher.load_state_dict(student.state_dict())
for p in teacher.parameters():
    p.requires_grad = False

dino_loss = DINOLoss(out_dim=OUT_DIM).to(device)
print(f'Student params: {sum(p.numel() for p in student.parameters() if p.requires_grad):,}')

## Training Loop

Each step:
1. Generate 2 global + N local crops per image
2. Forward all crops through student; forward only global crops through teacher
3. Compute cross-entropy between student and teacher soft distributions
4. Backprop through student only — teacher receives no gradients
5. Update teacher by EMA: `θ_t ← m·θ_t + (1−m)·θ_s`
6. Update center: `c ← λ·c + (1−λ)·mean(teacher_outputs)`

> The centering update is subtle — it tracks what the teacher "prefers" and subtracts it out, preventing any single dimension from dominating the output distribution.

In [ ]:
import os
os.makedirs('saved', exist_ok=True)

N_LOCAL = 4
GLOBAL_SIZE = 32
LOCAL_SIZE = 16
EPOCHS = 10
BATCH_SIZE = 128
BASE_LR = 3e-4
EMA_MOMENTUM_START = 0.996
EMA_MOMENTUM_END = 1.0

# Dataset
aug = DINOAugmentation(n_local=N_LOCAL, global_size=GLOBAL_SIZE, local_size=LOCAL_SIZE)
raw_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
train_ds = MultiCropDataset(raw_train, aug)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)

# Optimizer — AdamW with weight decay on non-bias/norm params
params = [p for p in student.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=BASE_LR, weight_decay=0.04)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_steps = EPOCHS * len(train_loader)

student.train()
global_step = 0

for epoch in range(EPOCHS):
    epoch_loss = 0.0

    for global_crops, local_crops, _ in train_loader:
        # Cosine annealing of EMA momentum
        tau = EMA_MOMENTUM_END - (EMA_MOMENTUM_END - EMA_MOMENTUM_START) * (
            math.cos(math.pi * global_step / total_steps) + 1) / 2
        global_step += 1

        # Move to device
        g1, g2 = global_crops[0].to(device), global_crops[1].to(device)
        locals_ = [lc.to(device) for lc in local_crops]

        # All views for student: 2 global + N local
        all_views = [g1, g2] + locals_

        # Student forward on all views
        student_out = [student(v) for v in all_views]

        # Teacher forward on global views only (no grad)
        with torch.no_grad():
            teacher_out = [teacher(g1), teacher(g2)]

        loss = dino_loss(student_out, teacher_out)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), max_norm=3.0)
        optimizer.step()

        # EMA update teacher
        with torch.no_grad():
            for ps, pt in zip(student.parameters(), teacher.parameters()):
                pt.data.mul_(tau).add_((1 - tau) * ps.data)

        epoch_loss += loss.item()

    scheduler.step()
    avg = epoch_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{EPOCHS}]  Loss: {avg:.4f}  EMA τ: {tau:.4f}')

torch.save(student.state_dict(), 'saved/dino_student.pt')
torch.save(teacher.state_dict(), 'saved/dino_teacher.pt')
print('Saved.')

## Linear Evaluation

Freeze the **teacher encoder** (it produces the most stable representations) and train a linear classifier on top. This protocol measures representation quality without any fine-tuning.

In [ ]:
# Freeze teacher encoder
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

linear_clf = nn.Linear(feat_dim, 10).to(device)
linear_opt = torch.optim.Adam(linear_clf.parameters(), lr=1e-3)

mean = [0.4914, 0.4822, 0.4465]
std  = [0.247,  0.243,  0.261]
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean, std)])

clf_train = torchvision.datasets.CIFAR10('./data', train=True,  transform=train_tf, download=False)
clf_test  = torchvision.datasets.CIFAR10('./data', train=False, transform=test_tf,  download=False)
clf_train_loader = DataLoader(clf_train, batch_size=256, shuffle=True,  num_workers=2)
clf_test_loader  = DataLoader(clf_test,  batch_size=256, shuffle=False, num_workers=2)

for ep in range(5):
    linear_clf.train()
    for imgs, labels in clf_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            feats = teacher.encoder(imgs)
        logits = linear_clf(feats)
        loss = F.cross_entropy(logits, labels)
        linear_opt.zero_grad()
        loss.backward()
        linear_opt.step()

    linear_clf.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in clf_test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            feats = teacher.encoder(imgs)
            preds = linear_clf(feats).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    print(f'Linear Eval Epoch {ep+1}/5 — Acc: {100*correct/total:.2f}%')

## Summary

| Component | Detail |
|-----------|--------|
| **Backbone** | ResNet-18 (CIFAR-adapted, 3×3 conv1, no maxpool) |
| **Head** | 3-layer MLP + weight-norm linear → K=2048 dims |
| **Multi-crop** | 2 global (32×32) + 4 local (16×16) |
| **Student temp** | τ_s = 0.1 (softer distribution) |
| **Teacher temp** | τ_t = 0.04 (sharper target) |
| **Centering** | EMA of teacher batch mean subtracted from teacher logits |
| **EMA momentum** | τ annealed 0.996 → 1.0 (cosine) |
| **Optimizer** | AdamW, lr=3e-4, weight_decay=0.04 |
| **Collapse prevention** | Centering (no negative pairs, no BN in backbone needed) |